# Ensemble Labeling: Vote, Mean, and LightGBM Strategies

This notebook generates ensemble labels for the OWS unlabeled corpus using three methods:
- **LightGBM (LGB)**: A trained gradient-boosting meta-model that combines raw probability outputs
  from four LLM annotators to predict Hate/Neutral labels.
- **Majority Vote**: A text is labeled Hate if at least 2 out of 4 LLMs assign a Hate probability above 0.5.
- **Mean Average**: The mean probability across the four LLMs is used; the label with the higher
  mean probability is assigned.

Input: The dataset `danghaidang-passau/Hate.2_labels_labeled`, which contains per-model probability
columns produced by `LLm_Labeling.ipynb`.

Output: New label columns `Lgb`, `Vote`, and `Mean` appended to the dataset for downstream training.

## 1. Imports and Setup

Import all required libraries and configure reproducibility seeds. Note that `FastLanguageModel`
from Unsloth is imported but not actively used in this ensemble-only notebook; it was kept for
compatibility with the LLM labeling codebase.

In [1]:
import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset
from datasets import load_dataset

from transformers import (
    set_seed,
)

from time import time
import pickle
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
from huggingface_hub import hf_hub_download

def_seed = 42

set_seed(def_seed)
np.random.seed(def_seed)
import random
random.seed(def_seed)

/home/dangdang/miniconda3/envs/ros_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Annotated Dataset

Load `danghaidang-passau/Hate.2_labels_labeled`, which contains the OWS corpus
with per-model probability columns from the four LLM annotators (produced by `LLm_Labeling.ipynb`).
Each row has columns like `unsloth/Qwen2.5-14B-Instruct-bnb-4bit_label_1`, etc.

The column definitions (`columns` list) map the full Unsloth model IDs to their respective
probability columns. This is used as the feature set for all three ensemble methods.

In [2]:
repo = "danghaidang-passau/HateOWS-dataset-LREC2026"
ds_annotated = load_dataset(repo, "annotatedOWS")    
df_2_label = ds_annotated["train"].to_pandas()

In [3]:
print("Total size of 2 labels data: ",df_2_label.shape[0])
print("Language counts ",df_2_label['language'].value_counts())

Total size of 2 labels data:  240647
Language counts  language
deu    125617
eng    108375
vie      6655
Name: count, dtype: int64


In [12]:
mstral7b = "mistral"
gemma9b = "gemma"
qwen14b = "qwen"
llama8B = "llama"
       

# Labeling for 2 labels: Hate  / Neutral. LightGBM, Vote, Mean Average

In [14]:
columns = [qwen14b, llama8B, mstral7b, gemma9b]

## 3. LightGBM Ensemble Labeling

Load the two pre-trained LightGBM models (`lgb_model_label_1.pkl` for Hate, `lgb_model_label_2.pkl`
for Neutral) from `Training/LightGbm_model.ipynb`. The models predict label probabilities using
the four LLM annotator outputs as features. A sample is labeled Hate (1) when the Hate probability
exceeds the Neutral probability and is also greater than 0.

In [15]:


with open('lgb_model_label_1.pkl', 'rb') as f:
    lgb_label_1 = pickle.load(f)

c1 = [x + "_prob_1" for x in columns]
X = df_2_label[c1]
val_preds = lgb_label_1.predict(X, num_iteration=lgb_label_1.best_iteration)
df_2_label["lgb_label_1"] = val_preds



with open('lgb_model_label_2.pkl', 'rb') as f:
    lgb_label_2 = pickle.load(f)

c2 = [x + "_prob_2" for x in columns]
X = df_2_label[c2]
val_preds = lgb_label_2.predict(X, num_iteration=lgb_label_2.best_iteration)
df_2_label["lgb_label_2"] = val_preds



index_1 = df_2_label['lgb_label_1'] > df_2_label['lgb_label_2']
index_2 = df_2_label['lgb_label_1'] > .0

index_all = index_1 & index_2

df_2_label['Lgb'] = 2
df_2_label.loc[index_all, 'Lgb'] = 1
df_2_label['Lgb'].value_counts()

Lgb
2    236232
1      4415
Name: count, dtype: int64

## 4. Majority Vote Ensemble Labeling

For each LLM, a sample is considered Hate if its predicted Hate-token probability exceeds 0.5.
A sample is assigned the Hate label if at least 2 out of 4 LLMs vote for Hate (majority threshold).
This is a threshold-free method that does not require any training data.

In [17]:
index_qwen = df_2_label[qwen14b + "_prob_1"] > .5
index_llama8b = df_2_label[llama8B+ "_prob_1"] > .5
index_gemma9b = df_2_label[gemma9b+ "_prob_1"] > .5
index_mstral7b = df_2_label[mstral7b+ "_prob_1"] > .5

from collections import Counter

index_all = index_mstral7b.astype(int) + index_gemma9b.astype(int) + index_llama8b.astype(int) + index_qwen.astype(int) 
index_all = index_all >= 2

df_2_label['Vote'] = 2
df_2_label.loc[index_all, 'Vote'] = 1
df_2_label['Vote'].value_counts()

Vote
2    235940
1      4707
Name: count, dtype: int64

## 5. Mean Average Ensemble Labeling

Compute the arithmetic mean of each label's probability across all four LLMs. The label with the
higher mean probability is assigned. This method treats all LLMs equally and does not require a
trained meta-model. It is equivalent to averaging the posterior probabilities of an ensemble.

In [19]:
c1 = [x + "_prob_1" for x in columns]
df_2_label["mean_label_1"] = df_2_label[c1].mean(axis=1)

c2 = [x + "_prob_2" for x in columns]
df_2_label["mean_label_2"] = df_2_label[c2].mean(axis=1)

index_1 = df_2_label['mean_label_1'] > df_2_label['mean_label_2']
index_2 = df_2_label['mean_label_1'] > .0
index_all = index_1 & index_2

df_2_label['Mean'] = 2
df_2_label.loc[index_all, 'Mean'] = 1
df_2_label['Mean'].value_counts()

Mean
2    236660
1      3987
Name: count, dtype: int64